# 07 — The editing experiment for InceptionV3 (extension)
Same three ablation curves as notebook 04, run on InceptionV3 so we can compare the editing pattern across architectures (cf. seankim's VGG16 result, where text-sorted recovered far less than sort-all):
- `text-sorted` — ablate MILAN-flagged text neurons, smallest val-acc impact first.
- `sort-all` — baseline: ablate any neuron, smallest val-acc impact first.
- `random` — averaged over random orderings.

Eval runs at **299×299** automatically (arch registry), so the ablation accuracies are computed on correctly-sized inputs.

In [ ]:
import os, sys, torch
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / 'milan'))
os.environ.setdefault('MILAN_DATA_DIR', str(Path.cwd().parent / 'data'))
os.environ.setdefault('MILAN_MODELS_DIR', str(Path.cwd().parent / 'models'))
os.environ.setdefault('MILAN_RESULTS_DIR', str(Path.cwd().parent / 'results'))

ARCH = 'inception_v3'
version_dir = Path(os.environ['MILAN_DATA_DIR']) / 'imagenet-spurious-text' / '50pct'
ckpt = Path(os.environ['MILAN_MODELS_DIR']) / f'{ARCH}_spurious.pth'
results = Path(os.environ['MILAN_RESULTS_DIR'])
dissect_dir = results / 'edit' / 'imagenet-spurious-text' / f'{ARCH}_spurious-50pct'
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
device

In [ ]:
# Per-unit importance (~3,936 ablation passes) + the three curves.
# Heavier than ResNet18; lower ablation_max / raise ablation_step to speed up.
from milan_repro.editing.evaluate import run as run_editing
run_editing(version_dir, ckpt, dissect_dir,
            results / f'descriptions_{ARCH}_annotated.csv',
            results / f'ablation_curve_{ARCH}.csv',
            arch=ARCH,
            n_random_trials=5, ablation_max=50, ablation_step=2,
            device=device)

In [ ]:
from milan_repro.figures.plot_fig7 import plot as plot_fig7
from milan_repro.figures.plot_fig8 import plot as plot_fig8
plot_fig7(dissect_dir, results / f'descriptions_{ARCH}_annotated.csv',
          results / 'figs' / f'fig7_{ARCH}.pdf')
plot_fig8(results / f'ablation_curve_{ARCH}.csv',
          results / 'figs' / f'fig8_{ARCH}.pdf',
          title='InceptionV3 robustness vs. ablation (MILAN Section 7, extension)')

In [ ]:
import pandas as pd
df = pd.read_csv(results / f'ablation_curve_{ARCH}.csv')
best = (df[df['mode'].isin(['text-sorted', 'sort-all'])]
        .groupby('mode')['adv_acc'].max())
baseline = df[df['mode'] == 'baseline'].iloc[0]
print('baseline adv acc:', baseline['adv_acc'])
print('best per mode:')
print(best)
print('\ntext-sorted recovery: +{:.1f} pt'.format(100*(best.get('text-sorted', float('nan')) - baseline['adv_acc'])))
print('sort-all   recovery: +{:.1f} pt'.format(100*(best.get('sort-all', float('nan')) - baseline['adv_acc'])))